# Hybrid GNN-LSTM for Portfolio P&L Prediction

This tutorial demonstrates the **HybridGnnRnn** model — a cutting-edge architecture combining **Graph Neural Networks** (GNN) for trade structure with **LSTM** for temporal P&L dynamics. The model predicts P&L of target trades given the portfolio graph and historical P&L of elementary trades.

## What You'll Learn

| Section | Topics |
|---------|--------|
| **Architecture** | GNN block (GraphSAGE), RNN block (LSTM), fusion (cross-attention + gating), target attention, projection |
| **Data** | Synthetic trade features, k-NN adjacency graph, P&L history, targets |
| **Training** | Model config, batching, training loop, callbacks |
| **Evaluation** | Metrics (MSE, MAE, R²), predicted vs actual, residuals, per-target error |
| **Visualisation** | Portfolio graph, feature distributions, training curves, evaluation dashboard |

## Prerequisites

```bash
pip install -r requirements.txt
```

Run the **Setup and imports** cell first. The first run may take a moment while TensorFlow loads.

In [ ]:
# Setup and imports — run this cell first
import sys
from pathlib import Path

def _find_project_root():
    path = Path.cwd()
    for _ in range(6):
        if (path / "src" / "m_learning").exists():
            return path
        if path.parent == path:
            break
        path = path.parent
    return Path.cwd().parents[2] if len(Path.cwd().parts) >= 3 else path

project_root = _find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

np.random.seed(42)
tf.random.set_seed(42)
print(f"Project root: {project_root}\nTensorFlow: {tf.__version__}")

---
## 1. Model Architecture

### 1.1 Purpose

The **HybridGnnRnn** model predicts P&L of **target trades** from:
1. **Trade structure:** A graph over all trades, where edges encode similarity (e.g. k-NN on trade features).
2. **Temporal P&L:** Historical P&L of **elementary trades** (building blocks) as a time series.

This is useful when target trades are new, illiquid, or complex — the model leverages relationships to similar trades and market dynamics.

### 1.2 Architecture Overview

$$
\text{Inputs} \;\xrightarrow{\text{GNN}}\; \text{Trade embeddings} \;\xrightarrow[\text{PnL history} \to \text{LSTM}]{\text{Fusion}}\; \text{Fused features} \;\xrightarrow{\text{Target Attention}}\; \text{Target features} \;\xrightarrow{\text{Projection}}\; \hat{y}
$$

| Block | Input | Output | Description |
|-------|-------|--------|-------------|
| **GnnBlock** | `trade_features` \\((T, F)\\), `adjacency` \\((T, T)\\) | \\((T, d_g)\\) | Message-passing (GraphSAGE) over the trade graph |
| **RnnBlock** | `pnl_history` \\((B, S, E)\\) | \\((B, d_r)\\) | LSTM over the P&L time series |
| **FusionLayer** | GNN emb \\((T, d_g)\\), RNN emb \\((B, d_r)\\), `adjacency` | \\((B, T, d_f)\\) | Cross-attention and gating to combine structural and temporal info |
| **TargetAttention** | Fused \\((B, T, d_f)\\), `adjacency`, `target_indices` | \\((B, N, d_a)\\) | Attend to fused features for each target trade |
| **TargetPnlOutput** | Attended \\((B, N, d_a)\\), `trade_features`, `target_indices` | \\((B, N)\\) | Per-target projection to P&L |

Where: \\(T\\) = trades, \\(F\\) = features, \\(B\\) = batch, \\(S\\) = timesteps, \\(E\\) = elementary trades, \\(N\\) = targets.

### 1.3 From Architecture to Code

| Component | Module | Config key |
|-----------|--------|------------|
| GNN block | `GnnBlock` | `gnn_model` |
| RNN block | `RnnBlock` | `rnn_model` |
| Fusion | `FusionLayer` | `fusion_model` |
| Target attention | `TargetAttentionLayer` | `attention_model` |
| Projection | `TargetPnlOutput` | `projection_model` |
| Top-level | `HybridGnnRnn` | `general` |

### 1.4 Input/Output Contract

**Inputs (dict):**
- `trade_features`: \\((T, F)\\) — numeric trade attributes (moneyness, TTM, delta, vega, product type).
- `adjacency_matrix`: \\((T, T)\\) — row-normalised adjacency (k-NN or similarity graph).
- `pnl_history`: \\((B, S, E)\\) — P&L of elementary trades over \\(S\\) time steps.
- `target_indices`: \\((N,)\\) — indices of target trades in \\([0, T)\\).
- `elementary_indices` (optional): \\((E,)\\) — indices of elementary trades.

**Output:**
- \\((B, N)\\) — predicted P&L for each target trade.

**Training objective:** MSE (or MAE) on predicted vs actual P&L.

---
## 2. Synthetic Data

We generate synthetic portfolio data:
- **Trade features:** moneyness, TTM, delta, vega, product type (one-hot).
- **Adjacency:** k-NN graph from trade features.
- **P&L history:** Random walk for elementary trades.
- **Targets:** Linear combination of final elementary P&L + noise.

In [ ]:
from src.m_learning.data.gnn_synthetic import (
    generate_synthetic_gnn_data,
    default_hybrid_model_config,
    SyntheticGnnData,
)

# Generate synthetic data
N_TRADES = 50
N_ELEMENTARY = 30
N_TARGETS = 10
N_SAMPLES = 500
N_TIMESTEPS = 20
K_NEIGHBOURS = 5

data = generate_synthetic_gnn_data(
    n_trades=N_TRADES,
    n_elementary=N_ELEMENTARY,
    n_targets=N_TARGETS,
    n_samples=N_SAMPLES,
    n_timesteps=N_TIMESTEPS,
    k_neighbours=K_NEIGHBOURS,
    noise_std=0.5,
    seed=42,
)

print(f"Trade features: {data.trade_features.shape}")
print(f"Adjacency: {data.adjacency_matrix.shape}")
print(f"P&L history: {data.pnl_history.shape}")
print(f"Targets: {data.targets.shape}")
print(f"Elementary indices: {data.elementary_indices[:5]}... (total {len(data.elementary_indices)})")
print(f"Target indices: {data.target_indices}")

### 2.1 Visualise: Trade Relationship Graph

We plot the portfolio graph: nodes = trades, edges = k-NN connections. Nodes are coloured by moneyness; target trades are shown with a distinct marker.

In [ ]:
from scipy.sparse.csgraph import laplacian
from scipy.linalg import eigh

def spectral_layout(adj, dim=2):
    """Compute 2D layout from adjacency using spectral embedding."""
    L = laplacian(adj, normed=True)
    # Small positive offset for numerical stability
    eigvals, eigvecs = eigh(L + 1e-6 * np.eye(L.shape[0]))
    return eigvecs[:, 1:dim+1]

pos = spectral_layout(data.adjacency_matrix, dim=2)
moneyness = data.trade_features[:, 0]

fig, ax = plt.subplots(figsize=(10, 8))
# Draw edges
adj = data.adjacency_matrix
for i in range(adj.shape[0]):
    for j in range(adj.shape[1]):
        if adj[i, j] > 0.01 and i < j:
            ax.plot([pos[i, 0], pos[j, 0]], [pos[i, 1], pos[j, 1]], 
                    c="gray", alpha=0.3, linewidth=0.5)

# Draw nodes
scatter = ax.scatter(pos[:, 0], pos[:, 1], c=moneyness, cmap="coolwarm", 
                     s=80, edgecolors="black", linewidths=0.5, zorder=5)
# Highlight targets
ax.scatter(pos[data.target_indices, 0], pos[data.target_indices, 1],
           s=200, facecolors="none", edgecolors="green", linewidths=2, zorder=6, label="Target trades")
# Highlight elementary
ax.scatter(pos[data.elementary_indices, 0], pos[data.elementary_indices, 1],
           marker="s", s=60, facecolors="none", edgecolors="blue", linewidths=1.5, zorder=6, label="Elementary trades")

plt.colorbar(scatter, ax=ax, label="Moneyness")
ax.set_title("Portfolio Trade Graph (k-NN adjacency)")
ax.set_xlabel("Spectral dim 1")
ax.set_ylabel("Spectral dim 2")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

### 2.2 Trade Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
features_to_plot = ["moneyness", "time_to_maturity", "delta", "vega"]
colors = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D"]

for ax, fname, color, idx in zip(axes.flat, features_to_plot, colors, range(4)):
    ax.hist(data.trade_features[:, idx], bins=15, color=color, alpha=0.7, edgecolor="white")
    ax.set_xlabel(fname.replace("_", " ").title())
    ax.set_ylabel("Count")
    ax.set_title(f"Distribution of {fname}")

plt.suptitle("Trade Feature Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 2.3 P&L History Sample

Heatmap of P&L over time for elementary trades (one sample).

In [ ]:
sample_idx = 0
pnl_sample = data.pnl_history[sample_idx]  # (n_timesteps, n_elementary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
im = axes[0].imshow(pnl_sample.T, aspect="auto", cmap="RdYlGn", origin="lower")
axes[0].set_xlabel("Time step")
axes[0].set_ylabel("Elementary trade index")
axes[0].set_title("P&L History Heatmap (sample 0)")
plt.colorbar(im, ax=axes[0], label="P&L")

# Line plot for a few elementary trades
for i in [0, 5, 10, 15, 20]:
    if i < pnl_sample.shape[1]:
        axes[1].plot(pnl_sample[:, i], label=f"Elem {data.elementary_indices[i]}")
axes[1].set_xlabel("Time step")
axes[1].set_ylabel("P&L")
axes[1].set_title("P&L Time Series (selected elementary trades)")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 3. Data Pipeline

We split into train/val and create `tf.data.Dataset` objects. The `BatchedHybridGnnRnn` wrapper handles the rank-2 vs batched inputs.

In [ ]:
from src.m_learning.models.gnn_rnn_hybrid.wrapper import (
    BatchedHybridGnnRnn,
    create_gnn_tf_dataset,
    train_val_split_gnn,
)

# Package inputs
gnn_inputs = data.to_gnn_inputs()
targets = data.targets

# Train/val split
train_inputs, train_targets, val_inputs, val_targets = train_val_split_gnn(
    gnn_inputs, targets, val_fraction=0.2, seed=42
)

print(f"Train samples: {train_inputs['pnl_history'].shape[0]}")
print(f"Val samples: {val_inputs['pnl_history'].shape[0]}")

# Create tf.data.Dataset
BATCH_SIZE = 32
train_ds = create_gnn_tf_dataset(train_inputs, train_targets, batch_size=BATCH_SIZE, shuffle=True, seed=42)
val_ds = create_gnn_tf_dataset(val_inputs, val_targets, batch_size=BATCH_SIZE, shuffle=False)

# Inspect one batch
for batch_inputs, batch_targets in train_ds.take(1):
    print(f"\nBatch shapes:")
    for k, v in batch_inputs.items():
        print(f"  {k}: {v.shape}")
    print(f"  targets: {batch_targets.shape}")

---
## 4. Model Configuration and Build

We use `default_hybrid_model_config()` for a minimal valid config, then build `BatchedHybridGnnRnn`.

In [ ]:
# Model config
model_config = default_hybrid_model_config(
    gnn_units=32,
    rnn_units=32,
    fusion_units=32,
    attention_units=32,
    projection_units=32,
    n_targets=N_TARGETS,
)

print("Model config keys:", list(model_config.keys()))
print("\nGNN config:", model_config["gnn_model"])
print("\nRNN config:", model_config["rnn_model"])

In [ ]:
# Build model
model = BatchedHybridGnnRnn(model_config, name="hybrid_pnl")

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"],
)

# Test forward pass
for batch_inputs, batch_targets in train_ds.take(1):
    test_out = model(batch_inputs, training=False)
    print(f"Forward pass output shape: {test_out.shape}")
    print(f"Expected: (batch, n_targets) = ({BATCH_SIZE}, {N_TARGETS})")

---
## 5. Training

Train the model with early stopping and learning rate reduction.

In [ ]:
EPOCHS = 50

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5, verbose=1
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

print(f"\nFinal train loss: {history.history['loss'][-1]:.6f}")
print(f"Final val loss: {history.history['val_loss'][-1]:.6f}")

---
## 6. Training Analytics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
epochs_range = range(1, len(history.history["loss"]) + 1)
axes[0].plot(epochs_range, history.history["loss"], label="Train loss", linewidth=2)
axes[0].plot(epochs_range, history.history["val_loss"], label="Val loss", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss (MSE)")
axes[0].set_title("Training and Validation Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE curve
axes[1].plot(epochs_range, history.history["mae"], label="Train MAE", linewidth=2)
axes[1].plot(epochs_range, history.history["val_mae"], label="Val MAE", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MAE")
axes[1].set_title("Training and Validation MAE")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. Evaluation and Visualisations

### 7.1 Predictions on Validation Set

In [ ]:
# Collect all predictions and actuals
all_preds = []
all_actuals = []
for batch_inputs, batch_targets in val_ds:
    preds = model(batch_inputs, training=False)
    all_preds.append(preds.numpy())
    all_actuals.append(batch_targets.numpy())

y_pred = np.concatenate(all_preds, axis=0).flatten()
y_true = np.concatenate(all_actuals, axis=0).flatten()
residuals = y_true - y_pred

print(f"Total predictions: {len(y_pred)}")

### 7.2 Summary Metrics Table

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
max_err = np.max(np.abs(residuals))
p50 = np.percentile(np.abs(residuals), 50)
p90 = np.percentile(np.abs(residuals), 90)
p99 = np.percentile(np.abs(residuals), 99)

metrics_df = pd.DataFrame({
    "Metric": ["MSE", "MAE", "R²", "Max Abs Error", "P50 Abs Error", "P90 Abs Error", "P99 Abs Error"],
    "Value": [mse, mae, r2, max_err, p50, p90, p99],
})
print("Validation Metrics:")
print(metrics_df.to_string(index=False))

### 7.3 Predicted vs Actual

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_true, y_pred, alpha=0.5, s=20, edgecolors="none")
lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
ax.plot(lims, lims, "r--", linewidth=2, label="45° line (perfect)")
ax.set_xlabel("Actual P&L")
ax.set_ylabel("Predicted P&L")
ax.set_title(f"Predicted vs Actual P&L (R² = {r2:.4f})")
ax.legend()
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7.4 Residual Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(residuals, bins=40, color="#2E86AB", alpha=0.7, edgecolor="white", density=True)
axes[0].axvline(0, color="red", linestyle="--", linewidth=2)
axes[0].set_xlabel("Residual (Actual - Predicted)")
axes[0].set_ylabel("Density")
axes[0].set_title("Residual Distribution")
axes[0].grid(True, alpha=0.3)

# Residual vs predicted
axes[1].scatter(y_pred, residuals, alpha=0.5, s=20, edgecolors="none")
axes[1].axhline(0, color="red", linestyle="--", linewidth=2)
axes[1].set_xlabel("Predicted P&L")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residual vs Predicted (Heteroscedasticity Check)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 7.5 Per-Target Error

### 7.6 Worst-Case Errors

In [ ]:
# Top-10 worst predictions
abs_errors = np.abs(y_true_2d - y_pred_2d)
flat_idx = np.argsort(abs_errors.flatten())[::-1][:10]
worst_sample_idx = flat_idx // N_TARGETS
worst_target_idx = flat_idx % N_TARGETS

worst_df = pd.DataFrame({
    "Sample": worst_sample_idx,
    "Target": [f"T{data.target_indices[t]}" for t in worst_target_idx],
    "Actual": [y_true_2d[s, t] for s, t in zip(worst_sample_idx, worst_target_idx)],
    "Predicted": [y_pred_2d[s, t] for s, t in zip(worst_sample_idx, worst_target_idx)],
    "Abs Error": [abs_errors[s, t] for s, t in zip(worst_sample_idx, worst_target_idx)],
})
print("Top-10 Worst Predictions:")
print(worst_df.to_string(index=False))

### 7.7 Distribution of Predictions vs Actuals

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max()), 40)
ax.hist(y_true, bins=bins, alpha=0.6, label="Actual", color="#2E86AB", density=True)
ax.hist(y_pred, bins=bins, alpha=0.6, label="Predicted", color="#F18F01", density=True)
ax.set_xlabel("P&L")
ax.set_ylabel("Density")
ax.set_title("Distribution of Actual vs Predicted P&L")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7.8 Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(16, 12))

# 1. Loss curve (top-left)
ax1 = fig.add_subplot(2, 3, 1)
ax1.plot(epochs_range, history.history["loss"], label="Train", linewidth=2)
ax1.plot(epochs_range, history.history["val_loss"], label="Val", linewidth=2)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss (MSE)")
ax1.set_title("Training Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Predicted vs actual (top-middle)
ax2 = fig.add_subplot(2, 3, 2)
ax2.scatter(y_true, y_pred, alpha=0.4, s=15, edgecolors="none")
ax2.plot(lims, lims, "r--", linewidth=2)
ax2.set_xlabel("Actual")
ax2.set_ylabel("Predicted")
ax2.set_title(f"Pred vs Actual (R²={r2:.3f})")
ax2.grid(True, alpha=0.3)

# 3. Residual distribution (top-right)
ax3 = fig.add_subplot(2, 3, 3)
ax3.hist(residuals, bins=30, color="#2E86AB", alpha=0.7, edgecolor="white", density=True)
ax3.axvline(0, color="red", linestyle="--", linewidth=2)
ax3.set_xlabel("Residual")
ax3.set_ylabel("Density")
ax3.set_title("Residual Distribution")
ax3.grid(True, alpha=0.3)

# 4. Per-target MAE (bottom-left)
ax4 = fig.add_subplot(2, 3, 4)
ax4.bar(range(len(per_target_mae)), per_target_mae, color=colors, edgecolor="black", linewidth=0.5)
ax4.set_xlabel("Target")
ax4.set_ylabel("MAE")
ax4.set_title("Per-Target MAE")
ax4.grid(True, alpha=0.3, axis="y")

# 5. Actual vs Predicted distribution (bottom-middle)
ax5 = fig.add_subplot(2, 3, 5)
ax5.hist(y_true, bins=30, alpha=0.6, label="Actual", color="#2E86AB", density=True)
ax5.hist(y_pred, bins=30, alpha=0.6, label="Predicted", color="#F18F01", density=True)
ax5.set_xlabel("P&L")
ax5.set_ylabel("Density")
ax5.set_title("P&L Distribution")
ax5.legend()
ax5.grid(True, alpha=0.3)

# 6. Metrics table (bottom-right)
ax6 = fig.add_subplot(2, 3, 6)
ax6.axis("off")
table_data = [
    ["MSE", f"{mse:.4f}"],
    ["MAE", f"{mae:.4f}"],
    ["R²", f"{r2:.4f}"],
    ["Max Error", f"{max_err:.4f}"],
    ["P90 Error", f"{p90:.4f}"],
]
table = ax6.table(cellText=table_data, colLabels=["Metric", "Value"], loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.5)
ax6.set_title("Summary Metrics", fontsize=12, fontweight="bold", pad=20)

plt.suptitle("Hybrid GNN-LSTM Model Evaluation Dashboard", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Conclusion

This tutorial demonstrated the **HybridGnnRnn** model for portfolio P&L prediction:

1. **Architecture:** GNN (GraphSAGE) captures trade relationships; LSTM captures temporal P&L dynamics; cross-attention fusion combines both; target-specific projection outputs P&L.

2. **Data:** Synthetic portfolio with k-NN adjacency graph and random-walk P&L history.

3. **Training:** Standard Keras workflow with early stopping and LR scheduling.

4. **Evaluation:** Comprehensive metrics (MSE, MAE, R², percentiles) and visualisations (pred vs actual, residuals, per-target error, distribution overlay, summary dashboard).

### Extensions

- **Real data:** Replace synthetic data with actual portfolio/trade data using `TradeAttributeEncoder` and `TradeGraphBuilder`.
- **Production pipelines:** Use `TrainingManager` for multi-stage training with checkpoints.
- **Attention visualisation:** Expose and plot attention weights from fusion and target attention layers.
- **Hyperparameter tuning:** Sweep over GNN/RNN units, layers, learning rate, etc.
- **Model comparison:** Compare against RNN-only baseline (`architecture: "rnn_only"`).